In [8]:
%pylab inline

%pylab is deprecated, use %matplotlib inline and import the required libraries.
Populating the interactive namespace from numpy and matplotlib


In [9]:
from numba import njit

In [10]:
# reduce the size of the index array where possible
max_index = N
bits_dtypes = [(8, uint8), (16, uint16), (32, uint32), (64, uint64)]
for bits, dt in bits_dtypes: 
    if 2**bits > max_index: break
print(bits, dt)

16 <class 'numpy.uint16'>


In [21]:
N=5120
t1 = array(uniform(0,1,N), dtype=float32)
t2 = array(uniform(0,1,N), dtype=float32)
t3 = array(uniform(0,0.6,N), dtype=float32)
idx = array([randint(0, N-1, 2223048), randint(0, N-1, 2223048), randint(0, N-1, 2223048)], dtype=int64).T

In [14]:
t2_coinc_window = 6
t3_coinc_window = 16

### Index array creation

In [15]:
@njit
def get_indices_jit(N, t2_coinc_window, t3_coinc_window, dtype=int64):
    idx = np.array([
        [i,j,k]
            for i in range(N)
                for j in range(max(i-t2_coinc_window, 0), min(N, i+t2_coinc_window+1))
                    for k in range(max(i-t3_coinc_window, 0), min(N, i+t3_coinc_window+1))
                ], dtype=dtype)
    return idx

def index_combinations(N, t2_coinc_window, t3_coinc_window, dtype=int64):
    """This messy funciton is equivalent to calling ..."""
    if 2*max(t2_coinc_window, t3_coinc_window) > N:
        raise NotImplementedError("analyzed time must be larger than "
            "the coincident window")
    # forgetting the tails at first
    largest_window = max(t2_coinc_window, t3_coinc_window)
    idx_1_mid = np.arange(N - 2*(t3_coinc_window+1), dtype=dtype).repeat( \
        (2*t2_coinc_window+1)*(2*t3_coinc_window+1) \
    ) + largest_window + 1
    idx_2_mid = idx_1_mid + np.tile(
        np.arange(-t2_coinc_window, t2_coinc_window+1, dtype=dtype).repeat(2*t3_coinc_window+1), 
        (N - 2*(t3_coinc_window+1))
    )
    idx_3_mid = idx_1_mid + np.tile(
        np.arange(-t3_coinc_window, t3_coinc_window+1, dtype=dtype), 
        (N - 2*(t3_coinc_window+1)) * (2*t2_coinc_window+1)
    )
    idx_1_ends, idx_2_ends, idx_3_ends = get_indices_jit(
        2*(t3_coinc_window+1), t2_coinc_window, t3_coinc_window, dtype=dtype).T
    # now get the tails
    n_start = int(len(idx_1_ends) / 2)
    idx_1 = np.concatenate((idx_1_ends[:n_start], idx_1_mid, idx_1_ends[-n_start:] + N - (2*t3_coinc_window + 2)))
    idx_2 = np.concatenate((idx_2_ends[:n_start], idx_2_mid, idx_2_ends[-n_start:] + N - (2*t3_coinc_window + 2)))
    idx_3 = np.concatenate((idx_3_ends[:n_start], idx_3_mid, idx_3_ends[-n_start:] + N - (2*t3_coinc_window + 2)))
    return np.array([idx_1, idx_2, idx_3]).T

In [22]:
%timeit idx = index_combinations(N, t2_coinc_window, t3_coinc_window, dt)

10.1 ms ± 66.9 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [23]:
%timeit idx = index_combinations(N, t2_coinc_window, t3_coinc_window)

15.6 ms ± 79.9 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [24]:
%timeit idx2 = get_indices_jit(N, t2_coinc_window, t3_coinc_window)

294 ms ± 4.39 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [172]:
idx = index_combinations(N, t2_coinc_window, t3_coinc_window, dt)
idx2 = get_indices_jit(N, t2_coinc_window, t3_coinc_window)

In [174]:
all(idx == idx2)

True

### Sum array

In [139]:
@njit
def three_det_sum_idx_jit(t1, t2, t3, idx):
    temp = array([
        t1[i] + t2[j] + t3[k]
            for i,j,k in idx
                ])
    return temp

@njit(parallel=True)
def three_det_sum_idx_jit_prange(t1, t2, t3, idx):
    temp=zeros(len(idx))
    for r in prange(len(idx)):
        temp[r] = t1[idx[r,0]] + t2[idx[r,1]] + t3[idx[r,2]]
    return temp

@njit(parallel=True)
def three_det_sum_idx_jit_prange_comp(t1, t2, t3, idx):
    temp = array([t1[idx[r,0]] + t2[idx[r,1]] + t3[idx[r,2]] for r in prange(len(idx))])
    return temp

def three_det_add_idx_array(t1, t2, t3, idx_1, idx_2, idx_3):
    out = t1[idx_1]
    out += t2[idx_2]
    return add(out, t3[idx_3], out=out)

def three_det_sum_idx_array(t1, t2, t3, idx_1, idx_2, idx_3):
    return t1[idx_1] + t2[idx_2] + t3[idx_3]

@njit(parallel=True)
def three_det_sum_idx_array_jit(t1, t2, t3, idx_1, idx_2, idx_3):
    return t1[idx_1] + t2[idx_2] + t3[idx_3]

@njit(parallel=True)
def three_det_sum_inplace_jit(t1, t2, t3, idx_1, idx_2, idx_3):
    out = t1[idx_1]
    out += t2[idx_2]
    out += t3[idx_3]
    return out

@njit(parallel=True)
def three_det_sum_idx_array_take_inplace_jit(t1, t2, t3, idx_1, idx_2, idx_3):
    out = take(t1,idx_1)
    out += take(t2,idx_2)
    out += take(t3,idx_3)
    return out

@njit(parallel=True)
def three_det_sum_idx_array_take_jit(t1, t2, t3, idx_1, idx_2, idx_3):
    return take(t1,idx_1) + take(t2,idx_2) + take(t3,idx_3)

In [134]:
out = three_det_sum_idx_jit(t1, t2, t3, idx)

In [140]:
%timeit out = three_det_sum_idx_jit(t1, t2, t3, idx)
%timeit out = three_det_sum_idx_jit_prange_comp(t1, t2, t3, idx)
%timeit out = three_det_sum_idx_jit_prange(t1, t2, t3, idx)
%timeit out = three_det_sum_idx_array(t1, t2, t3, idx_1, idx_2, idx_3)
%timeit out = three_det_add_idx_array(t1, t2, t3, idx_1, idx_2, idx_3)
%timeit out = three_det_sum_idx_array_jit(t1, t2, t3, idx_1, idx_2, idx_3)
%timeit out = three_det_sum_inplace_jit(t1, t2, t3, idx_1, idx_2, idx_3)
%timeit out = three_det_sum_idx_array_take_inplace_jit(t1, t2, t3, idx_1, idx_2, idx_3)
%timeit out = three_det_sum_idx_array_take_jit(t1, t2, t3, idx_1, idx_2, idx_3)

/Users/camill/miniconda3/envs/igwn-py38/lib/python3.8/site-packages/numba/core/typed_passes.py:331: NumbaPerformanceWarning: 
The keyword argument 'parallel=True' was specified but no transformation for parallel execution was possible.

To find out why, try turning on parallel diagnostics, see https://numba.pydata.org/numba-doc/latest/user/parallel.html#diagnostics for help.

File "../../../../../../var/folders/2n/44xg31f92bd49rgkf9bsy4j80000gq/T/ipykernel_39303/717079672.py", line 1:
<source missing, REPL/exec in use?>

  warnings.warn(errors.NumbaPerformanceWarning(msg,


3.64 ms ± 21.7 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)
1.76 ms ± 143 µs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)
2.18 ms ± 289 µs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)
6.07 ms ± 29.8 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)
6.06 ms ± 15.2 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)
3.87 ms ± 127 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)
4.12 ms ± 507 µs per loop (mean ± std. dev. of 7 runs, 1 loop each)
6.16 ms ± 665 µs per loop (mean ± std. dev. of 7 runs, 1 loop each)
5.22 ms ± 265 µs per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [133]:
%timeit out = three_det_sum_idx_jit(t1, t2, t3, idx)
%timeit out = three_det_sum_idx_jit_prange_comp(t1, t2, t3, idx)
%timeit out = three_det_sum_idx_jit_prange(t1, t2, t3, idx)
%timeit out = three_det_sum_idx_array(t1, t2, t3, idx_1, idx_2, idx_3)
%timeit out = three_det_add_idx_array(t1, t2, t3, idx_1, idx_2, idx_3)
%timeit out = three_det_sum_idx_array_jit(t1, t2, t3, idx_1, idx_2, idx_3)
%timeit out = three_det_sum_inplace_jit(t1, t2, t3, idx_1, idx_2, idx_3)
%timeit out = three_det_sum_idx_array_take_inplace_jit(t1, t2, t3, idx_1, idx_2, idx_3)
%timeit out = three_det_sum_idx_array_take_jit(t1, t2, t3, idx_1, idx_2, idx_3)

1.67 ms ± 5.77 µs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)
1.75 ms ± 98 µs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)
1.89 ms ± 365 µs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)
5.95 ms ± 10.3 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)
5.96 ms ± 15.3 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)
3.69 ms ± 31.5 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)
3.73 ms ± 47.7 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)
5.8 ms ± 831 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)
4.96 ms ± 218 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [87]:
%timeit out = three_det_sum_idx_jit(t1, t2, t3, idx)
%timeit out = three_det_sum_idx_jit_prange_comp(t1, t2, t3, idx)
%timeit out = three_det_sum_idx_jit_prange(t1, t2, t3, idx)
%timeit out = three_det_sum_idx_array(t1, t2, t3, idx_1, idx_2, idx_3)
%timeit out = three_det_add_idx_array(t1, t2, t3, idx_1, idx_2, idx_3)
%timeit out = three_det_sum_idx_array_jit(t1, t2, t3, idx_1, idx_2, idx_3)
%timeit out = three_det_sum_inplace_jit(t1, t2, t3, idx_1, idx_2, idx_3)
%timeit out = three_det_sum_idx_array_take_inplace_jit(t1, t2, t3, idx_1, idx_2, idx_3)
%timeit out = three_det_sum_idx_array_take_jit(t1, t2, t3, idx_1, idx_2, idx_3)

4.46 ms ± 30.1 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)
2.75 ms ± 728 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)
3.04 ms ± 320 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)
6.88 ms ± 21.5 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)
6.9 ms ± 21.8 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)
4.66 ms ± 331 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)
4.84 ms ± 465 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)
6.29 ms ± 791 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)
5.6 ms ± 106 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [75]:
%timeit out = three_det_sum_idx_jit(t1, t2, t3, idx)
%timeit out = three_det_sum_idx_array(t1, t2, t3, idx_1, idx_2, idx_3)
%timeit out = three_det_add_idx_array(t1, t2, t3, idx_1, idx_2, idx_3)
%timeit out = three_det_sum_idx_array_jit(t1, t2, t3, idx_1, idx_2, idx_3)
%timeit out = three_det_sum_inplace_jit(t1, t2, t3, idx_1, idx_2, idx_3)
%timeit out = three_det_sum_idx_array_take_inplace_jit(t1, t2, t3, idx_1, idx_2, idx_3)
%timeit out = three_det_sum_idx_array_take_jit(t1, t2, t3, idx_1, idx_2, idx_3)

4.52 ms ± 48 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)
6.82 ms ± 9.3 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)
6.82 ms ± 16.8 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)
4.14 ms ± 5.92 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)
4.14 ms ± 3.85 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)
5.08 ms ± 3.76 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)
5.08 ms ± 631 ns per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [26]:
len(out)

2223048